# 64. Showcase 노트북 — ToxGuard 데모GitHub 저장소를 처음 보는 사람이 별도 설명 없이 이 노트북 하나만실행해봐도 프로젝트가 무엇을 하는지 바로 이해할 수 있도록 만든시연용 노트북입니다.

In [ ]:
!pip install rdkit -q!pip install openai -q

In [ ]:
from google.colab import userdatatoken = userdata.get('GITHUB_TOKEN')!git clone https://{token}@github.com/Dec32th/laidd-2026.git%cd /content/laidd-2026!pwd

# ToxGuard 데모Tox21 데이터셋의 분자에서 독성 구조(toxicophore)를 자동 진단하고,화학적으로 타당한 치환을 제안·검증하는 파이프라인입니다.- RDKit 규칙(BRENK/PAINS) 기반 진단- 42개 규칙에 대한 치환 후보 라이브러리 (ChEMBL/Guide to  Pharmacology 실조회로 검증된 27건의 선례 포함)- LLM(Qwen) proposer-critic 토의로 화학적으로 의심스러운 치환을  검증- 단백질 도킹(AutoDock Vina/Meeko), 합성용이성(SA Score), 활성보존  지표를 판단 근거로 결합JUMP AI 2026 공모전(팀명 MorForge) 제출작을 기반으로 지속 개발중입니다. 자세한 내용은 [docs/README.md](docs/README.md),전체 개발 이력은 [docs/CHANGELOG.md](docs/CHANGELOG.md),방법론적 발견 사례는 [docs/discovery_patterns.md](docs/discovery_patterns.md)를 참고하세요.

In [ ]:
import syssys.path.append('.')from src.tools.toxicophore_detector import detect_toxicophoresfrom src.tools.replacement_library import get_replacement_candidatesfrom src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memoryfrom src.tools.precedent_library import get_precedentsfrom src.tools.visualization import visualize_fix_processprint("✅ 핵심 모듈 로드 완료")

## 사례 1: 긴 지방족 사슬(Aliphatic_long_chain) 진단 및 치환BRENK 필터가 지적하는 대표적 구조 중 하나로, 4개 이상의 연속된비고리형 탄소 사슬을 가진 분자를 다룹니다. 아래 분자는 폴리에틸렌글리콜형 계면활성제 계열 구조입니다.

In [ ]:
demo_smiles_1 = "CCCCCCCCCCCCCCCCOCCO"  # 긴 알킬 사슬 + 에테르problems = detect_toxicophores(demo_smiles_1)print(f"입력 분자: {demo_smiles_1}\n")print(f"진단된 문제 {len(problems)}건:")for p in problems:    print(f"  - {p['rule_name']}")

In [ ]:
clear_failure_memory()result = iterative_fix_loop(demo_smiles_1, max_iterations=10, candidate_idx=0)print(f"최종 상태: {result['status']}")print(f"최종 분자: {result.get('final_smiles', '')}\n")result_viz = visualize_fix_process(result)result_viz['image_2d']

Aliphatic_long_chain 규칙은 개발 초기에 흥미로운 발견으로 이어진사례입니다. BRENK 필터의 실제 SMARTS 정의가 원소 종류와 무관한순수 위상학적 패턴(연속된 degree-2 원자 4개)이라는 것을 직접분석으로 확인했고, 이에 따라 에테르 삽입 대신 메틸 분기 전략으로전환해 완전 해결에 이르렀습니다. 자세한 내용은[discovery_patterns.md](docs/discovery_patterns.md)를 참고하세요.

## 사례 2: LLM 근거 오귀속 방지 — Caveat 메커니즘개발 중 발견한 중요한 문제: 도킹 검증 결과를 LLM(critic)에게 판단근거로 제공했을 때, 그 도킹이 "이 특정 분자의 실제 작용 표적"이아니라 단순 벤치마크였을 뿐인데도 critic이 "이 분자는 이 표적이다"라고 근거 없이 단정하는 경향이 관찰되었습니다.이를 막기 위해 각 도킹 표적 등록에 `caveat` 필드를 추가해, 표적특이성 근거가 없는 경우를 LLM 프롬프트에 명시적으로 알립니다.

In [ ]:
import importlibfrom src.tools.docking import DOCKING_TARGETStarget = DOCKING_TARGETS['Michael_acceptor_1']print("표적:", target.get('target_name', target.get('pdb_id', '?')))print("\ncaveat 필드:")print(target['caveat'])

이처럼 정량적 도구(도킹 등)의 출력을 LLM에게 근거로 제공할 때는,그 근거가 실제로 이 분자에 대해 무엇을 말해주고 무엇을 말해주지않는지를 명시적으로 제약하지 않으면 과확신에 찬 잘못된 설명이생성될 수 있습니다. 이 발견을 포함해, 개발 과정에서 반복적으로나타난 "비교/검증 실험이 시스템 결함을 발견하는 도구로 기능한"네 가지 사례는 [docs/discovery_patterns.md](docs/discovery_patterns.md)에 정리되어 있습니다.

## 전체 파이프라인 성능 (valid set, 1173개 분자)규칙 기반 파이프라인과 LLM 토의를 포함한 파이프라인을 비교한최종 수치입니다. 토의 파이프라인의 수치가 더 낮은 것은 성능저하가 아니라 의도된 트레이드오프입니다 — 화학적으로 의심스러운자동 치환을 막아 표면적 성공률은 낮아지지만, 각 성공의 신뢰도는올라갑니다.

In [ ]:
!apt-get install -y fonts-nanum -qq > /dev/nullimport matplotlib.font_manager as fmimport matplotlib.pyplot as pltfont_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'fm.fontManager.addfont(font_path)plt.rc('font', family='NanumGothic')plt.rcParams['axes.unicode_minus'] = False

In [ ]:
data = {    "예선 제출": 52.0,    "규칙 기반\n(현재)": 74.5,    "토의 파이프라인\n(현재)": 68.5,}fig, ax = plt.subplots(figsize=(6, 4))bars = ax.bar(data.keys(), data.values(), color=["#999999", "#4C72B0", "#55A868"])ax.set_ylabel("완전 해결 비율 (%)")ax.set_ylim(0, 100)ax.set_title("ToxGuard 파이프라인 성능 비교")for bar, val in zip(bars, data.values()):    ax.text(bar.get_x() + bar.get_width()/2, val + 2, f"{val}%", ha='center')plt.tight_layout()plt.show()

## 더 알아보기- 전체 개발 이력(노트북 1-63): [docs/CHANGELOG.md](docs/CHANGELOG.md)- 프로젝트 개요: [docs/README.md](docs/README.md)- 방법론적 발견 사례 4건: [docs/discovery_patterns.md](docs/discovery_patterns.md)- 알려진 한계: [docs/limitations.md](docs/limitations.md)- 공유결합 도킹(Meeko+AutoDock-GPU) 성공 사례:  [docs/meeko_covalent_docking_progress.md](docs/meeko_covalent_docking_progress.md)JUMP AI 2026 공모전(팀명 MorForge) 제출작으로 시작해, 예선 결과와무관하게 지속 개발 중인 프로젝트입니다.